# Introduction: The Question {#sec-intro}

Why does a one-day lift pass at Aspen cost three times more than one at Val di
Fiemme? Is it just about the size of the mountain, or is something else going
on?

Skiing is a global industry worth billions, yet ski pass prices vary wildly
between resorts that — at least from a skier's point of view — offer
comparable experiences. This project investigates **what factors actually
determine the price of a ski lift pass worldwide**.

To answer this, we follow a structured approach inspired by detective work:
we identify four potential "suspects" — different categories of factors that
*might* explain pricing — and we interrogate each of them with the appropriate
statistical tests. Finally, we put all the evidence together in a multivariate
model that tells us which suspect is really responsible.

## The Four Suspects

1.  **The Mountain Itself** — Do bigger, higher resorts charge more? (structural factors)
2.  **Geography and Wealth** — Does location and country wealth drive pricing? (geographic and economic factors)
3.  **Services and Positioning** — Do extras like snowparks or summer skiing justify a premium? (commercial factors)
4.  **The Actual Snow** — Does real snow cover correlate with price? (environmental factors)

Each section of this notebook tackles one suspect, and the final synthesis
brings everything together with a multivariate regression model.

------------------------------------------------------------------------

# Setup and Data Loading {#sec-setup}

## Libraries

In [1]:
# Data import and manipulation
install.packages("rio")
install.packages("dplyr")
install.packages("rcompanion")
install.packages("sjmisc")
library(rio)
library(dplyr)

# Statistical analysis
library(rcompanion)   # for Cramer's V

# Optional, for nicer tables in the final notebook
# library(knitr)
# library(kableExtra)

NameError: name 'install' is not defined

## Working Directory

In [ ]:
# Set your working directory to the project folder
# setwd("path/to/project")

## Importing the Data

The main dataset comes from the *Ski Resorts* dataset on Kaggle
(Ulrik Thyge Pedersen, 2023), containing 499 resorts worldwide with 25
variables covering location, structural characteristics, services, and price.

The file uses Latin-1 encoding because resort names include accented
characters (French, Austrian, Italian resorts).

In [ ]:
resorts <- import("data/resorts.csv", encoding = "Latin-1")

# Quick overview
str(resorts)
nrow(resorts)
ncol(resorts)

The companion `snow.csv` file contains monthly snow cover data for
geographic coordinates in 2022. We will load it later in Section 4.

------------------------------------------------------------------------

# Data Cleaning and Preparation {#sec-cleaning}

Real-world datasets are never as clean as they look at first glance. Before
analyzing prices, we need to detect anomalies, handle missing values, and
create derived variables that will support our investigation.

## Exploring the Variables

In [ ]:
summary(resorts)
table(resorts$Continent)

The dataset is heavily skewed toward Europe (360 of 499 resorts), with North
America (98) as the second-largest group. Oceania (10) and South America (7)
are underrepresented — a limitation we will keep in mind throughout the
analysis.

## Detecting Missing Values

Apparent NAs are zero, but missing data is often **hidden as zeros or
placeholder strings** in real datasets.

In [ ]:
# Check explicit NAs across all columns
apply(resorts, 2, function(x) sum(is.na(x)))

To detect hidden zeros, we apply the same pattern restricted to numeric
columns (mixing types would coerce everything to character and break the
comparison):

In [ ]:
colonne_num <- sapply(resorts, is.numeric)
apply(resorts[, colonne_num], 2, function(x) sum(x == 0, na.rm = TRUE))

**Interpretation of the zeros found:**

-   `Price`: 9 zeros → impossible (a ski pass cannot be free) → **masked NAs**
-   `Total lifts`: 1 zero → impossible (a ski resort needs lifts) → **masked NA**
-   `Beginner / Intermediate / Difficult slopes`: plausible (a resort may lack a difficulty tier)
-   `Snow cannons` (226), `Gondola lifts` (175): plausible (small resorts often lack these)
-   `Longest run` (212): suspicious but ambiguous — we keep them for now

In [ ]:
# Convert suspicious zeros to NA
resorts$Price <- ifelse(resorts$Price == 0, NA, resorts$Price)
resorts$`Total lifts` <- ifelse(resorts$`Total lifts` == 0, NA, resorts$`Total lifts`)

# Verify
apply(resorts, 2, function(x) sum(is.na(x)))

## Cleaning the `Season` Variable

The `Season` variable is particularly problematic, with 31 unique values
and 27 entries marked as `"Unknown"` (clearly placeholder NAs).

In [ ]:
table(resorts$Season)
length(unique(resorts$Season))

We recode it into 4 meaningful macro-categories + NA:

-   **Winter_Only**: typical winter season (Nov–May)
-   **Summer_Only**: summer months only (glaciers / Southern Hemisphere)
-   **Multi_Season**: open both in winter and summer
-   **Year_Round**: open all year
-   **NA**: previously "Unknown"

In [ ]:
# Define value vectors for each category
seasons_winter <- c("December - April", "November - April", "December - March",
                    "November - May", "October - May", "December - May",
                    "October - April", "April", "October - June", "September - June",
                    "September - May", "March", "December", "July - April",
                    "November - June", "September - April", "November - March")

seasons_summer <- c("June - October", "June - September", "July - September",
                    "May - September", "July", "May - October", "May",
                    "July - October")

seasons_multi <- c("November - May, June - August",
                   "December - April, June - August, October - November",
                   "October - November, December - May, June - October")

# Nested ifelse with %in% operator
resorts$Season <- ifelse(resorts$Season == "Unknown", NA,
                  ifelse(resorts$Season == "Year-round", "Year_Round",
                  ifelse(resorts$Season %in% seasons_multi, "Multi_Season",
                  ifelse(resorts$Season %in% seasons_summer, "Summer_Only",
                  ifelse(resorts$Season %in% seasons_winter, "Winter_Only", NA)))))

table(resorts$Season, useNA = "ifany")

## Creating Derived Variables

Two new variables that will support multiple analyses downstream:

In [ ]:
# Vertical drop = highest - lowest point
resorts <- mutate(resorts, Vertical_Drop = `Highest point` - `Lowest point`)

# Altitude band (categorical, useful for ANOVA)
resorts$Altitude_Band <- cut(resorts$`Highest point`,
                             breaks = c(0, 1500, 2500, 3500, 5000),
                             labels = c("Low", "Medium", "High", "Very_High"))

table(resorts$Altitude_Band)

## Loading External Datasets

To enrich the analysis beyond what's in the original Kaggle dataset, we
integrate two external sources:

-   **Per capita GDP** (World Bank, 2022) — proxy for national wealth and
    purchasing power
-   **Cost of Living Index** (Numbeo, 2022) — captures local price levels
    in tourism destinations

In [ ]:
# Load external data (CSV files prepared from official sources)
# gdp_data <- import("data/gdp_per_capita.csv")
# col_data <- import("data/cost_of_living_index.csv")

# Merge on Country
# resorts <- merge(resorts, gdp_data, by = "Country", all.x = TRUE)
# resorts <- merge(resorts, col_data, by = "Country", all.x = TRUE)

# Verify which countries failed to match (likely due to naming mismatches)
# unique(resorts$Country[is.na(resorts$GDP_per_capita)])

**Note for the team:** before merging, check country name alignment between
sources. Common mismatches: "United States" vs "USA", "Czech Republic" vs
"Czechia", "South Korea" vs "Korea, Rep.".

------------------------------------------------------------------------

# Suspect 1 — The Mountain Itself {#sec-suspect1}

**Assigned to: \[Name 1\]**

**Hypothesis:** larger and higher resorts charge higher prices because they
offer more skiing terrain and require more infrastructure to maintain.

## Descriptive Overview

In [ ]:
# Price distribution
summary(resorts$Price)
hist(resorts$Price,
     main = "Distribution of ski pass prices",
     xlab = "Price (EUR)",
     col = "lightblue")

The mean (51.4 EUR) is higher than the median (45 EUR), indicating a
**right-skewed distribution** — a few premium resorts pull the average up.

## Correlation Tests

**Hypotheses for all correlation tests:**

-   H0: the correlation coefficient = 0 (no linear relationship)
-   H1: the correlation coefficient ≠ 0 (a linear relationship exists)

In [ ]:
# Price vs maximum altitude
cor.test(resorts$`Highest point`, resorts$Price, use = "pairwise.complete.obs")

# Price vs total number of slopes
cor.test(resorts$`Total slopes`, resorts$Price, use = "pairwise.complete.obs")

# Price vs total number of lifts
cor.test(resorts$`Total lifts`, resorts$Price, use = "pairwise.complete.obs")

# Price vs vertical drop
cor.test(resorts$Vertical_Drop, resorts$Price, use = "pairwise.complete.obs")

**Findings:**

-   `Highest point`: r ≈ 0.41, p \< 0.001 → **moderate positive correlation**
-   `Total slopes`: r ≈ 0.31, p \< 0.001 → **moderate positive correlation**
-   `Total lifts`: r ≈ 0.10, p ≈ 0.02 → weak but significant
-   `Vertical_Drop`: r ≈ 0.17, p \< 0.001 → weak but significant

The strongest single predictor is **maximum altitude**, not size per se.

## ANOVA: Price by Altitude Band

**Hypotheses:**

-   H0: mean price is equal across all altitude bands
-   H1: at least one altitude band has a significantly different mean price

In [ ]:
anova_altitude <- aov(Price ~ Altitude_Band, data = resorts)
summary(anova_altitude)
TukeyHSD(anova_altitude)

## Preliminary Conclusion for Suspect 1

> Structural factors — particularly maximum altitude and number of slopes —
> show a clear positive relationship with price. However, the strength is
> moderate at best, suggesting that "bigger = more expensive" is only part
> of the story.

------------------------------------------------------------------------

# Suspect 2 — Geography and Wealth {#sec-suspect2}

**Assigned to: \[Name 2\]**

**Hypothesis:** prices vary systematically by region not just because of
geography, but because of the economic context of each country (purchasing
power, cost of living, tourism market structure).

## Price by Continent

In [ ]:
resorts %>%
  group_by(Continent) %>%
  summarise(
    N = n(),
    Mean_Price = round(mean(Price, na.rm = TRUE), 2),
    Median_Price = median(Price, na.rm = TRUE),
    SD_Price = round(sd(Price, na.rm = TRUE), 2)
  )

## ANOVA: Price by Continent

**Hypotheses:**

-   H0: mean price is equal across all continents
-   H1: at least one continent has a significantly different mean price

In [ ]:
anova_continent <- aov(Price ~ Continent, data = resorts)
summary(anova_continent)
TukeyHSD(anova_continent)

## Correlation with National Wealth

Once the external GDP and Cost of Living data are merged in Section 2.6:

In [ ]:
# Correlation: price vs GDP per capita
# cor.test(resorts$GDP_per_capita, resorts$Price, use = "pairwise.complete.obs")

# Correlation: price vs Cost of Living Index
# cor.test(resorts$Cost_of_Living, resorts$Price, use = "pairwise.complete.obs")

## Chi-Square: Continent vs Price Category

**Hypotheses:**

-   H0: continent and price category are independent
-   H1: continent and price category are associated

In [ ]:
# Create a categorical price variable using terciles
# resorts$Price_Category <- cut(resorts$Price,
#                               breaks = quantile(resorts$Price, probs = c(0, 1/3, 2/3, 1), na.rm = TRUE),
#                               labels = c("Low", "Medium", "High"),
#                               include.lowest = TRUE)

# Contingency table
# tab_geo <- table(resorts$Continent, resorts$Price_Category)
# tab_geo
# addmargins(tab_geo)
# round(prop.table(tab_geo, margin = 1), 3) * 100

# Chi-square test
# chisq.test(tab_geo)

# Strength of association
# cramerV(tab_geo)

## Preliminary Conclusion for Suspect 2

> *To be written based on results.* Expected: North American resorts charge
> substantially more, and this gap is partly — but probably not fully —
> explained by differences in national wealth and cost of living.

------------------------------------------------------------------------

# Suspect 3 — Services and Commercial Positioning {#sec-suspect3}

**Assigned to: \[Name 3\]**

**Hypothesis:** additional services (snowparks, night skiing, summer skiing,
family-friendly facilities) command a price premium because they target
specific customer segments willing to pay more.

## Service Variables Distribution

In [ ]:
table(resorts$Snowparks)
table(resorts$Nightskiing)
table(resorts$`Summer skiing`)
table(resorts$`Child friendly`)

## T-tests: Price by Each Service

**Hypotheses (for each service):**

-   H0: mean price is the same for resorts with vs without the service
-   H1: mean price differs between resorts with vs without the service

In [ ]:
# Snowpark
t.test(resorts$Price[resorts$Snowparks == "Yes"],
       resorts$Price[resorts$Snowparks == "No"])

# Night skiing
t.test(resorts$Price[resorts$Nightskiing == "Yes"],
       resorts$Price[resorts$Nightskiing == "No"])

# Summer skiing
t.test(resorts$Price[resorts$`Summer skiing` == "Yes"],
       resorts$Price[resorts$`Summer skiing` == "No"])

# Child friendly
t.test(resorts$Price[resorts$`Child friendly` == "Yes"],
       resorts$Price[resorts$`Child friendly` == "No"])

## ANOVA: Price by Seasonality

Using the recoded `Season` variable. Note: `Year_Round` and `Multi_Season`
groups are small (4 each), so this analysis is exploratory.

**Hypotheses:**

-   H0: mean price is equal across all seasonality types
-   H1: at least one seasonality type has a significantly different mean price

In [ ]:
anova_season <- aov(Price ~ Season, data = resorts)
summary(anova_season)
TukeyHSD(anova_season)

## Chi-Square: Service Availability by Continent

**Hypotheses:**

-   H0: service availability is independent of continent
-   H1: service availability is associated with continent

In [ ]:
# Example: summer skiing by continent
tab_summer <- table(resorts$Continent, resorts$`Summer skiing`)
tab_summer
addmargins(tab_summer)
round(prop.table(tab_summer, margin = 1), 3) * 100

chisq.test(tab_summer)
cramerV(tab_summer)

## Preliminary Conclusion for Suspect 3

> *To be written based on results.* Expected: services like snowparks and
> night skiing add a measurable premium, but summer skiing is geographically
> determined (Southern Hemisphere / glaciers) rather than a pure positioning
> choice.

------------------------------------------------------------------------

# Suspect 4 — The Actual Snow {#sec-suspect4}

**Assigned to: \[Name 4\]**

**Hypothesis:** resorts with more reliable natural snow cover can charge
higher prices because they offer a better and more predictable skiing
experience.

## Loading the Snow Dataset

In [ ]:
# snow <- import("data/snow.csv")
# str(snow)
# nrow(snow)
# 
# # snow.csv contains monthly snow values for grid points (Lat/Lon) in 2022
# # Each row: Month, Latitude, Longitude, Snow (snow cover indicator)

## Aggregating Snow Data per Resort

The snow dataset has snow values for a geographic grid; we need to associate
each resort with nearby snow measurements. The simplest approach is to round
latitude and longitude to a common precision and aggregate.

In [ ]:
# Round coordinates to 0.25 degrees (matches typical grid spacing)
# snow_agg <- snow %>%
#   mutate(Lat_round = round(Latitude * 4) / 4,
#          Lon_round = round(Longitude * 4) / 4) %>%
#   group_by(Lat_round, Lon_round) %>%
#   summarise(Mean_Snow = mean(Snow, na.rm = TRUE),
#             Max_Snow = max(Snow, na.rm = TRUE),
#             .groups = "drop")
# 
# resorts <- resorts %>%
#   mutate(Lat_round = round(Latitude * 4) / 4,
#          Lon_round = round(Longitude * 4) / 4) %>%
#   left_join(snow_agg, by = c("Lat_round", "Lon_round"))
# 
# # How many resorts got matched?
# sum(!is.na(resorts$Mean_Snow))

## Correlation: Snow vs Price

**Hypotheses:**

-   H0: there is no linear relationship between snow cover and price
-   H1: there is a linear relationship between snow cover and price

In [ ]:
# cor.test(resorts$Mean_Snow, resorts$Price, use = "pairwise.complete.obs")

## Correlation: Snow vs Altitude

A natural validity check — we expect strong positive correlation:

In [ ]:
# cor.test(resorts$Mean_Snow, resorts$`Highest point`, use = "pairwise.complete.obs")

## ANOVA: Snow by Continent

**Hypotheses:**

-   H0: mean snow cover is equal across all continents
-   H1: at least one continent has significantly different mean snow cover

In [ ]:
# anova_snow_continent <- aov(Mean_Snow ~ Continent, data = resorts)
# summary(anova_snow_continent)
# TukeyHSD(anova_snow_continent)

## Preliminary Conclusion for Suspect 4

> *To be written based on results.* Expected: actual snow cover is more
> strongly tied to altitude and latitude than to price directly, suggesting
> that "snow reliability" is already partly priced in through altitude.

------------------------------------------------------------------------

# Synthesis: The Verdict {#sec-synthesis}

So far, each section has examined one factor at a time. But in reality,
these factors are **entangled**: North American resorts charge more — but is
it because they are American, or because they are *also* larger, higher, and
better serviced?

To answer this final question, we build a **multiple linear regression
model** that combines all relevant predictors and tells us how much each
factor contributes *after accounting for the others*.

## The Multivariate Model

In [ ]:
# Placeholder: full model to be defined together
# 
# model_full <- lm(Price ~ `Highest point` + `Total slopes` + Vertical_Drop +
#                          Continent + Altitude_Band +
#                          Snowparks + Nightskiing + `Summer skiing` +
#                          Season,
#                  data = resorts)
# 
# summary(model_full)

## Model Diagnostics

In [ ]:
# par(mfrow = c(2, 2))
# plot(model_full)
# par(mfrow = c(1, 1))

## Interpretation

*To be filled in based on results.* Key questions to answer:

-   Which variables remain significant predictors when controlling for the others?
-   How much of the variance in price does the model explain (Adjusted R²)?
-   Are there factors that looked important in bivariate tests but lose significance in the full model? (suggests confounding)
-   Are there factors that gain importance? (suggests suppression effects)

------------------------------------------------------------------------

# Conclusions {#sec-conclusions}

## Answer to the Research Question

*To be written.* A one-sentence synthesis along the lines of:

> "Ski pass prices are driven primarily by \[\_\_\_\], while \[\_\_\_\] matters less than initial intuition would suggest."

## The Surprise

*To be written.* Every good analysis has at least one counterintuitive
finding. Identify and explain the most interesting one.

## Limitations

-   The dataset contains 9 prices masked as zero (handled as NA)
-   The `Season` variable required heavy recoding from 31 raw values
-   Oceania (10) and South America (7) are heavily underrepresented
-   Prices are a snapshot in time (2023) — no temporal dimension
-   Correlation does not imply causation: the regression identifies
    *associations*, not causal mechanisms
-   External data sources (GDP, Cost of Living) are at the country level,
    not at the resort level — this introduces ecological inference issues

## Practical Implications

*To be written.* Who could use these findings?

-   Tourists choosing a destination on a budget
-   New resorts setting their pricing strategy
-   Tourism boards positioning their offer
-   Investors evaluating ski resort acquisitions

## Open Questions

*To be written.* What new questions does this analysis raise that we could
not answer? Climate change impacts, multi-year price evolution, customer
behavior data, etc.

------------------------------------------------------------------------

# Data Sources and References

-   **Ski Resorts dataset**: Pedersen, U. T. (2023). *Ski Resorts*.
    Kaggle. <https://www.kaggle.com/datasets/ulrikthygepedersen/ski-resorts>
-   **Snow Cover dataset**: included in the Kaggle package above (`snow.csv`)
-   **GDP per capita**: World Bank Open Data, 2022.
    <https://data.worldbank.org/indicator/NY.GDP.PCAP.CD>
-   **Cost of Living Index**: Numbeo, 2022. <https://www.numbeo.com/cost-of-living/>

------------------------------------------------------------------------

# Appendix: Session Info

In [ ]:
sessionInfo()